# Проверка что код применим к другим конфигурациям

In [1]:
import yaml
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

## Загрузка конфигурации сети из YAML

In [2]:
yaml_str = """
servers:
  - id: chain1
    mu: 7.159457094969063
    queue_type: FIFO
    lambda: 3.579728547484531
  - id: chain2
    mu: 2.367157068548567
    queue_type: PS
  - id: chain3
    mu: 6.084057042335131
    queue_type: FIFO
  - id: chain4
    mu: 1.197904448724397
    queue_type: PS
transitions:
  - from: chain1
    to: exit
    probability: 0.7998614202260583
  - from: chain1
    to: chain3
    probability: 0.1579390742381085
  - from: chain1
    to: chain2
    probability: 0.04219950553583315
  - from: chain2
    to: exit
    probability: 0.4890477719917962
  - from: chain2
    to: chain1
    probability: 0.2312069757232702
  - from: chain2
    to: chain4
    probability: 0.2797452522849335
  - from: chain3
    to: exit
    probability: 0.7830293746341181
  - from: chain3
    to: chain1
    probability: 0.1126989522087693
  - from: chain3
    to: chain2
    probability: 0.1042716731571126
  - from: chain4
    to: exit
    probability: 0.6284635396431909
  - from: chain4
    to: chain1
    probability: 0.3205993083362966
  - from: chain4
    to: chain2
    probability: 0.05093715202051252
"""
config = yaml.safe_load(yaml_str)

In [3]:
# config

## Парсинг результатов симуляции

In [4]:
sim_text = """
Server 'chain1'
  Type: FIFO
  Cost: 26628.9
  Jobs: 3751
  Avg queue time: 0.145195
  Avg service time: 0.139562
  Utilization: 0.523677
Server 'chain2'
  Type: PS
  Cost: 3801.72
  Jobs: 225
  Avg queue time: 0.0462857
  Avg service time: 0.397827
  Utilization: 0.089511
Server 'chain3'
  Type: FIFO
  Cost: 19507.9
  Jobs: 603
  Avg queue time: 0.0264107
  Avg service time: 0.173573
  Utilization: 0.104665
Server 'chain4'
  Type: PS
  Cost: 1717.49
  Jobs: 57
  Avg queue time: 0.0160039
  Avg service time: 0.825079
  Utilization: 0.0470295

Total system cost: 51656
Total tasks: 3589
Mean time: 0.366639
"""
def parse_simulation(text):
    server_blocks = text.strip().split("Server")
    data = {}
    for block in server_blocks[1:]:
        # id
        m_id = re.search(r"'([^']+)'", block)
        if not m_id:
            continue
        srv_id = m_id.group(1)
        # key-value pairs
        kvs = {}
        for line in block.splitlines():
            m = re.match(r"\s*(.+?):\s*(.*)", line)
            if m:
                k, v = m.group(1).strip(), m.group(2)
                kvs[k] = v
        # convert to proper types
        parsed = {}
        for k, v in kvs.items():
            if k.strip() == 'Type':
                parsed['Type'] = v.strip()
            elif k.strip() == 'Cost':
                parsed['Cost'] = float(v)
            elif k.strip() == 'Jobs':
                parsed['Jobs'] = int(v)
            elif k.strip() == 'Avg queue time':
                parsed['Avg_queue_time'] = float(v)
            elif k.strip() == 'Avg service time':
                parsed['Avg_service_time'] = float(v)
            elif k.strip() == 'Utilization':
                parsed['Utilization'] = float(v)
        data[srv_id] = parsed
    # parse global metrics
    total_cost = re.search(r"Total system cost:\s*([-\d\.]+)", text).group(1)
    total_tasks = re.search(r"Total tasks:\s*(\d+)", text).group(1)
    mean_time = re.search(r"Mean time:\s*([-\d\.]+)", text).group(1)
    return data, float(total_cost), int(total_tasks), float(mean_time)
sim_data, total_cost, total_tasks, mean_sojourn = parse_simulation(sim_text)
print(f"Total system cost:  {total_cost:.3f}")
print(f"Total tasks:        {total_tasks:.3f}")
print(f"Mean time:          {mean_sojourn:.3f}")

Total system cost:  51656.000
Total tasks:        3589.000
Mean time:          0.367


In [5]:
pd.DataFrame.from_dict(sim_data, orient='index')

,Type,Cost,Jobs,Avg_queue_time,Avg_service_time,Utilization
chain1,FIFO,26628.90,3751,0.145195,0.139562,0.523677
chain2,PS,3801.72,225,0.046286,0.397827,0.089511
chain3,FIFO,19507.90,603,0.026411,0.173573,0.104665
chain4,PS,1717.49,57,0.016004,0.825079,0.047030


In [6]:
for srv in config['servers']:
    sid = srv['id']
    data = sim_data.get(sid)
    if data:
        srv['Cost'] = data['Cost']
        srv['Jobs'] = data['Jobs']
        srv['Avg_queue_time'] = data['Avg_queue_time']
        srv['Avg_service_time'] = data['Avg_service_time']
        srv['Utilization'] = data['Utilization']
# config

## Основные метрики сети
### Пропускная способность и средняя задержка
*Пропускная способность* – это количество задач, обработанных за единицу времени  
(средний throughput). Поскольку симуляция проводилась в течение `max time = 1000` секунд,  
$$
\lambda_{\text{net}}=\frac{\text{Total tasks}}{\text{Sim. time}}
$$
*Средняя задержка* – это уже заданный показатель mean sojourn time.

In [7]:
sim_time = 1000  # как указано в выводе симуляции
throughput_net = total_tasks / sim_time
mean_delay_net = mean_sojourn
print(f"Throughput (tasks/s):  {throughput_net:.3f}")
print(f"Mean delay (s):        {mean_delay_net:.5f}")
# Взвешенный параметр Φ
phi_net = throughput_net / mean_delay_net
print(f"Φ (throughput/delay):  {phi_net:.3f}")

Throughput (tasks/s):  3.589
Mean delay (s):        0.36664
Φ (throughput/delay):  9.789


### Таблица по серверам

In [8]:
df_srv = pd.DataFrame.from_dict(sim_data, orient='index')
df_srv['mu'] = df_srv.index.map(lambda x: next((s['mu'] for s in config['servers'] if s['id'] == x), np.nan))
df_srv['lambda_ext'] = None
# только chain1 имеет внешнюю интенсивность λ в конфиге
for srv in config['servers']:
    if 'lambda' in srv:
        df_srv.at[srv['id'], 'lambda_ext'] = srv['lambda']
df_srv[['Type', 'Cost', 'Jobs', 'Avg_queue_time', 'Avg_service_time', 'Utilization', 'mu', 'lambda_ext']]

,Type,Cost,Jobs,Avg_queue_time,Avg_service_time,Utilization,mu,lambda_ext
chain1,FIFO,26628.90,3751,0.145195,0.139562,0.523677,7.159457,3.579729
chain2,PS,3801.72,225,0.046286,0.397827,0.089511,2.367157,None
chain3,FIFO,19507.90,603,0.026411,0.173573,0.104665,6.084057,None
chain4,PS,1717.49,57,0.016004,0.825079,0.047030,1.197904,None


## Оценка стоимости добавления новых серверов
Для каждого типа сервера вычислим **затраты на единицу скорости обслуживания**:
$$
c_{\mu} = \frac{\text{Cost}}{\mu}
$$
Это позволяет понять, какой тип узла наиболее экономичен.

In [9]:
cost_mu = {}
for srv in config['servers']:
    cost_mu[srv['id']] = srv['Cost'] / srv['mu'] 
cost_mu

{'chain1': 3719.4021343758145,
 'chain2': 1606.0277750521395,
 'chain3': 3206.3966304485284,
 'chain4': 1433.7454058450905}

### Бюджет

In [10]:
budget = 0.10 * total_cost   # 10 % от исходной стоимости
print(f"Бюджет на новые серверы: {budget:.2f}")


Бюджет на новые серверы: 5165.60


## Анализ текущей сети: уравнения трафика и узкие места

In [11]:
import numpy as np
from copy import deepcopy

def solve_traffic(servers, transitions):
    """
    Решает уравнения трафика для открытой сети Джексона.
    Возвращает словарь lambda_eff (интенсивности), матрицу переходов P (только между серверами),
    внешние интенсивности lambda_ext, списки ids и словарь idx.
    """
    n = len(servers)
    ids = [s['id'] for s in servers]
    idx = {id_: i for i, id_ in enumerate(ids)}

    # Матрица переходов P (n x n) между серверами
    P = np.zeros((n, n))
    # Внешний поток
    lambda_ext = np.zeros(n)
    for s in servers:
        if 'lambda' in s:
            lambda_ext[idx[s['id']]] = s['lambda']

    for t in transitions:
        from_id = t['from']
        to_id = t['to']
        if to_id == 'exit':
            continue  # выход из сети, не добавляем в матрицу переходов
        i = idx[from_id]
        j = idx[to_id]
        P[i, j] = t['probability']

    # Решаем (I - P^T) * lambda = lambda_ext
    A = np.eye(n) - P.T
    lambda_eff = np.linalg.solve(A, lambda_ext)

    # Словарь для удобства
    lambda_dict = {ids[i]: lambda_eff[i] for i in range(n)}
    return lambda_dict, P, lambda_ext, ids, idx

def compute_metrics(servers, lambda_eff):
    """
    Вычисляет утилизацию ρ, среднее время пребывания E[T],
    коэффициент посещения V (относительно внешнего потока),
    вклад в общее время и общее E[T_sys].
    """
    metrics = {}
    total_lambda_ext = sum(s.get('lambda', 0) for s in servers)
    for s in servers:
        sid = s['id']
        mu = s['mu']
        lam = lambda_eff[sid]
        rho = lam / mu
        E_T = 1.0 / (mu - lam) if mu > lam else np.inf
        # Среднее время обслуживания и очереди
        E_S = 1.0 / mu
        E_W = E_T - E_S
        metrics[sid] = {
            'lambda': lam,
            'mu': mu,
            'rho': rho,
            'E_T': E_T,
            'E_S': E_S,
            'E_W': E_W,
        }
    # Коэффициенты посещения и общее время
    V = {}
    for s in servers:
        sid = s['id']
        V[sid] = metrics[sid]['lambda'] / total_lambda_ext
    E_T_sys = sum(V[sid] * metrics[sid]['E_T'] for sid in metrics)
    for sid in metrics:
        metrics[sid]['V'] = V[sid]
        metrics[sid]['contrib'] = V[sid] * metrics[sid]['E_T']
    return metrics, E_T_sys, total_lambda_ext

# Применяем к исходной конфигурации
lambda_eff, P, lambda_ext, ids, idx = solve_traffic(config['servers'], config['transitions'])
metrics, E_T_sys, total_lambda_ext = compute_metrics(config['servers'], lambda_eff)

print("=== Текущие эффективные интенсивности λ_i ===")
for sid in metrics:
    print(f"{sid}: λ = {metrics[sid]['lambda']:.4f}, ρ = {metrics[sid]['rho']:.4f}, E[T] = {metrics[sid]['E_T']:.4f}, V = {metrics[sid]['V']:.4f}, вклад = {metrics[sid]['contrib']:.4f}")
print(f"\nОбщая интенсивность внешнего потока: {total_lambda_ext:.4f}")
print(f"Теоретическое среднее время пребывания E[T_sys] = {E_T_sys:.4f} (симуляция: {mean_sojourn:.4f})")

=== Текущие эффективные интенсивности λ_i ===
chain1: λ = 3.7169, ρ = 0.5192, E[T] = 0.2905, V = 1.0383, вклад = 0.3016
chain2: λ = 0.2212, ρ = 0.0935, E[T] = 0.4660, V = 0.0618, вклад = 0.0288
chain3: λ = 0.5870, ρ = 0.0965, E[T] = 0.1819, V = 0.1640, вклад = 0.0298
chain4: λ = 0.0619, ρ = 0.0517, E[T] = 0.8803, V = 0.0173, вклад = 0.0152

Общая интенсивность внешнего потока: 3.5797
Теоретическое среднее время пребывания E[T_sys] = 0.3755 (симуляция: 0.3666)


In [12]:
# Определим узел с наибольшим вкладом в задержку
bottleneck = max(metrics.items(), key=lambda x: x[1]['contrib'])
print(f"\nБутылочное горлышко: {bottleneck[0]} (вклад {bottleneck[1]['contrib']:.4f} из {E_T_sys:.4f})")


Бутылочное горлышко: chain1 (вклад 0.3016 из 0.3755)


## Модель стоимости серверов

In [13]:
def cost_model(servers, queue_type):
    """
    По серверам заданного типа оценивает коэффициенты c и α в cost = c * μ^α.
    Возвращает (c, alpha).
    """
    # Собираем пары (mu, cost) для серверов нужного типа
    data = [(s['mu'], s['Cost']) for s in servers if s['queue_type'] == queue_type]
    if len(data) < 2:
        raise ValueError(f"Недостаточно серверов типа {queue_type} для построения модели")
    mu1, cost1 = data[0]
    mu2, cost2 = data[1]
    alpha = np.log(cost2 / cost1) / np.log(mu2 / mu1)
    c = cost1 / (mu1 ** alpha)
    return c, alpha

# Для FIFO и PS
c_fifo, alpha_fifo = cost_model(config['servers'], 'FIFO')
c_ps, alpha_ps = cost_model(config['servers'], 'PS')
print(f"FIFO: cost = {c_fifo:.2f} * μ^{alpha_fifo:.3f}")
print(f"PS:   cost = {c_ps:.2f} * μ^{alpha_ps:.3f}")

# Бюджет уже вычислен
print(f"\nБюджет на новые серверы: {budget:.2f} (10% от {total_cost:.2f})")

FIFO: cost = 617.94 * μ^1.912
PS:   cost = 1391.26 * μ^1.167

Бюджет на новые серверы: 5165.60 (10% от 51656.00)


## Подбор вариантов стратегий

In [14]:
from copy import deepcopy

# ---------- Вспомогательные функции ----------
def get_cost_params(queue_type):
    if queue_type == 'FIFO':
        return c_fifo, alpha_fifo
    else:
        return c_ps, alpha_ps


def compute_server_params(budget, queue_type, Lambda, mu_A, alpha_fallback=0.5):
    c, alpha_m = get_cost_params(queue_type)
    if budget <= 0:
        return 0.0, 0.0
    mu = (budget / c) ** (1.0 / alpha_m)
    alpha = (mu - mu_A + Lambda) / (2 * Lambda)
    alpha = np.clip(alpha, 0.0, 1.0)
    if alpha < 0.1 or alpha > 0.9:
        alpha = alpha_fallback
        mu = mu_A + (2 * alpha - 1) * Lambda
        if mu <= 0:
            mu = 0.1
        cost_tmp = c * (mu ** alpha_m)
        if cost_tmp > budget:
            mu = (budget / c) ** (1.0 / alpha_m)
            alpha = (mu - mu_A + Lambda) / (2 * Lambda)
            alpha = np.clip(alpha, 0.0, 1.0)
    return mu, alpha


def add_parallel_server(config, target, mu_new, alpha, queue_type='PS'):
    new_servers = deepcopy(config['servers'])
    new_transitions = deepcopy(config['transitions'])
    target_server = next(s for s in new_servers if s['id'] == target)
    lambda_ext_orig = target_server.get('lambda', 0.0)
    new_id = f"{target}_p"
    new_server = {'id': new_id, 'mu': mu_new, 'queue_type': queue_type}
    if lambda_ext_orig > 0:
        new_server['lambda'] = alpha * lambda_ext_orig
        target_server['lambda'] = (1 - alpha) * lambda_ext_orig
    new_servers.append(new_server)
    for trans in new_transitions:
        if trans['to'] == target:
            p_old = trans['probability']
            trans['probability'] = (1 - alpha) * p_old
            new_transitions.append({'from': trans['from'], 'to': new_id, 'probability': alpha * p_old})
    outgoing = [t for t in config['transitions'] if t['from'] == target]
    for t in outgoing:
        new_transitions.append({'from': new_id, 'to': t['to'], 'probability': t['probability']})
    return {'servers': new_servers, 'transitions': new_transitions}


def add_two_parallel_same_node_mixed(config, target, mu_list, alpha_list, type_list):
    """Добавляет два сервера параллельно одному узлу."""
    new_servers = deepcopy(config['servers'])
    new_transitions = deepcopy(config['transitions'])
    target_server = next(s for s in new_servers if s['id'] == target)
    lambda_ext_orig = target_server.get('lambda', 0.0)
    new_ids = []
    for i, (mu, alpha, qtype) in enumerate(zip(mu_list, alpha_list, type_list)):
        new_id = f"{target}_p{i + 1}"
        new_server = {'id': new_id, 'mu': mu, 'queue_type': qtype}
        if lambda_ext_orig > 0:
            new_server['lambda'] = alpha * lambda_ext_orig
        new_servers.append(new_server)
        new_ids.append((alpha, new_id))
    if lambda_ext_orig > 0:
        target_server['lambda'] = (1 - alpha_list[0] - alpha_list[1]) * lambda_ext_orig
    for trans in new_transitions:
        if trans['to'] == target:
            p_old = trans['probability']
            trans['probability'] = (1 - alpha_list[0] - alpha_list[1]) * p_old
            for alpha, new_id in new_ids:
                new_transitions.append({'from': trans['from'], 'to': new_id, 'probability': alpha * p_old})
    outgoing = [t for t in config['transitions'] if t['from'] == target]
    for alpha, new_id in new_ids:
        for t in outgoing:
            new_transitions.append({'from': new_id, 'to': t['to'], 'probability': t['probability']})
    return {'servers': new_servers, 'transitions': new_transitions}


def evaluate_E_T(config):
    lambda_eff, _, _, _, _ = solve_traffic(config['servers'], config['transitions'])
    _, E_T_sys, _ = compute_metrics(config['servers'], lambda_eff)
    return E_T_sys


def evaluate_cost(config):
    """Вычисляет общую стоимость конфигурации."""
    total = 0
    for s in config['servers']:
        c, alpha_m = get_cost_params(s['queue_type'])
        total += c * (s['mu'] ** alpha_m)
    return total


# ---------- Универсальный сценарий для двух серверов ----------
def try_two_servers(targets, budgets):
    """
    targets: список узлов (длина 1 или 2)
    budgets: список бюджетов для каждого сервера (длина len(targets))
             Если len(targets)==1, то budgets содержит два значения (бюджеты для двух серверов)
             Если len(targets)==2, то budgets содержит по одному бюджету на узел.
    Возвращает (E_T, config, description) или None.
    Также печатает информацию о каждом проверенном варианте.
    """
    if len(targets) == 1:
        target = targets[0]
        Lambda = metrics[target]['lambda']
        mu_A = metrics[target]['mu']
        budget1, budget2 = budgets
        best_local = None
        # Перебираем все комбинации типов для двух серверов
        type_combos = [('PS', 'PS'), ('FIFO', 'FIFO'), ('PS', 'FIFO'), ('FIFO', 'PS')]
        for qtype1, qtype2 in type_combos:
            c1, am1 = get_cost_params(qtype1)
            c2, am2 = get_cost_params(qtype2)
            mu1 = (budget1 / c1) ** (1.0 / am1) if budget1 > 0 else 0.0
            mu2 = (budget2 / c2) ** (1.0 / am2) if budget2 > 0 else 0.0
            if mu1 <= 0 or mu2 <= 0:
                continue
            # Оптимальные доли
            D = (mu1 + mu2 + mu_A - Lambda) / 3.0
            alpha1 = (mu1 - D) / Lambda
            alpha2 = (mu2 - D) / Lambda
            if alpha1 < 0 or alpha2 < 0 or (1 - alpha1 - alpha2) < 0:
                alpha1 = alpha2 = 1.0 / 3.0
            else:
                alpha1 = np.clip(alpha1, 0.01, 0.98)
                alpha2 = np.clip(alpha2, 0.01, 0.98)
                if 1 - alpha1 - alpha2 < 0:
                    s = alpha1 + alpha2
                    alpha1 = alpha1 / s * 0.98
                    alpha2 = alpha2 / s * 0.98
            config_new = add_two_parallel_same_node_mixed(config, target, [mu1, mu2], [alpha1, alpha2],
                                                          [qtype1, qtype2])
            E_T = evaluate_E_T(config_new)
            cost_new = evaluate_cost(config_new)
            added = cost_new - total_cost
            desc = f"2 сервера к {target}: {qtype1}(μ={mu1:.4f},α={alpha1:.4f}) и {qtype2}(μ={mu2:.4f},α={alpha2:.4f})"
            print(f"  {desc}: E[T]={E_T:.4f}, доп. стоимость={added:.2f} (бюджет {budget:.2f})")
            if best_local is None or E_T < best_local[0]:
                best_local = (E_T, config_new, desc)
        return best_local
    elif len(targets) == 2:
        target1, target2 = targets
        budget1, budget2 = budgets
        Lambda1, mu_A1 = metrics[target1]['lambda'], metrics[target1]['mu']
        Lambda2, mu_A2 = metrics[target2]['lambda'], metrics[target2]['mu']
        best_local = None
        # Получаем возможные варианты для каждого узла
        candidates1 = []
        for qtype in ['PS', 'FIFO']:
            mu, alpha = compute_server_params(budget1, qtype, Lambda1, mu_A1)
            if mu > 0:
                candidates1.append((mu, alpha, qtype))
        candidates2 = []
        for qtype in ['PS', 'FIFO']:
            mu, alpha = compute_server_params(budget2, qtype, Lambda2, mu_A2)
            if mu > 0:
                candidates2.append((mu, alpha, qtype))
        for mu1, alpha1, t1 in candidates1:
            for mu2, alpha2, t2 in candidates2:
                temp_conf = add_parallel_server(config, target1, mu1, alpha1, queue_type=t1)
                temp_conf = add_parallel_server(temp_conf, target2, mu2, alpha2, queue_type=t2)
                E_T = evaluate_E_T(temp_conf)
                cost_new = evaluate_cost(temp_conf)
                added = cost_new - total_cost
                desc = f"2 сервера: {target1}({t1},μ={mu1:.4f}) и {target2}({t2},μ={mu2:.4f})"
                print(f"  {desc}: E[T]={E_T:.4f}, доп. стоимость={added:.2f} (бюджет {budget:.2f})")
                if best_local is None or E_T < best_local[0]:
                    best_local = (E_T, temp_conf, desc)
        return best_local
    else:
        return None


# ---------- Основной блок (расширенный перебор) ----------
c_fifo, alpha_fifo = cost_model(config['servers'], 'FIFO')
c_ps, alpha_ps = cost_model(config['servers'], 'PS')
print(f"Модели стоимости: FIFO: {c_fifo:.2f}*μ^{alpha_fifo:.3f}, PS: {c_ps:.2f}*μ^{alpha_ps:.3f}")

candidates = [(sid, metrics[sid]['contrib']) for sid in metrics if metrics[sid]['lambda'] > 1e-6]
candidates.sort(key=lambda x: x[1], reverse=True)
print("Узлы-кандидаты (вклад в общую задержку):")
for sid, contrib in candidates:
    print(f"  {sid}: contrib={contrib:.4f}, ρ={metrics[sid]['rho']:.4f}, E[T]={metrics[sid]['E_T']:.4f}")

best_E_T = E_T_sys
best_config = deepcopy(config)
best_desc = "исходная конфигурация"

print("\n=== Сценарий 1: один сервер к каждому узлу-кандидату ===")
for target, _ in candidates:
    Lambda_t = metrics[target]['lambda']
    mu_A_t = metrics[target]['mu']
    for qtype in ['PS', 'FIFO']:
        mu, alpha = compute_server_params(budget, qtype, Lambda_t, mu_A_t)
        if mu > 0 and alpha > 0.01:  # только значимые доли
            config1 = add_parallel_server(config, target, mu, alpha, queue_type=qtype)
            E_T = evaluate_E_T(config1)
            cost1 = evaluate_cost(config1)
            added = cost1 - total_cost
            desc = f"1 сервер ({qtype}) к {target} (μ={mu:.4f}, α={alpha:.4f})"
            print(f"  {desc}: E[T]={E_T:.4f}, доп. стоимость={added:.2f}")
            if E_T < best_E_T - 1e-9:  # улучшение с учётом погрешности
                best_E_T, best_config, best_desc = E_T, config1, desc

print("\n=== Сценарий 2: два сервера ===")
# 2a: два сервера к одному узлу (перебираем все узлы-кандидаты)
print("  Подсценарий 2a: два сервера к одному узлу, бюджет поровну")
for target, _ in candidates:
    budget_half = budget / 2.0
    res = try_two_servers([target], [budget_half, budget_half])
    if res:
        E_T, cfg, desc = res
        if E_T < best_E_T - 1e-9:
            best_E_T, best_config, best_desc = E_T, cfg, desc

# 2b: два сервера к двум разным узлам (перебираем все пары, не только топ-2)
if len(candidates) >= 2:
    print("  Подсценарий 2b: два сервера к двум разным узлам, бюджет пропорционально вкладу")
    # Перебираем все уникальные пары (i<j)
    for i in range(len(candidates)):
        for j in range(i+1, len(candidates)):
            target1, contrib1 = candidates[i]
            target2, contrib2 = candidates[j]
            total_contrib = contrib1 + contrib2
            budget1 = budget * (contrib1 / total_contrib)
            budget2 = budget * (contrib2 / total_contrib)
            # Пропускаем, если один из бюджетов слишком мал
            if budget1 < 1e-6 or budget2 < 1e-6:
                continue
            res = try_two_servers([target1, target2], [budget1, budget2])
            if res:
                E_T, cfg, desc = res
                if E_T < best_E_T - 1e-9:
                    best_E_T, best_config, best_desc = E_T, cfg, desc

print("\n" + "=" * 60)
print("ЛУЧШИЙ ВАРИАНТ")
print("=" * 60)
print(best_desc)
print(f"Теоретическое E[T_sys]: {E_T_sys:.4f} → {best_E_T:.4f} (улучшение {(E_T_sys - best_E_T) / E_T_sys * 100:.2f}%)")

new_config = best_config

Модели стоимости: FIFO: 617.94*μ^1.912, PS: 1391.26*μ^1.167
Узлы-кандидаты (вклад в общую задержку):
  chain1: contrib=0.3016, ρ=0.5192, E[T]=0.2905
  chain3: contrib=0.0298, ρ=0.0965, E[T]=0.1819
  chain2: contrib=0.0288, ρ=0.0935, E[T]=0.4660
  chain4: contrib=0.0152, ρ=0.0517, E[T]=0.8803

=== Сценарий 1: один сервер к каждому узлу-кандидату ===
  1 сервер (PS) к chain2 (μ=2.3672, α=0.5000): E[T]=0.3740, доп. стоимость=3801.73
  1 сервер (FIFO) к chain2 (μ=2.3672, α=0.5000): E[T]=0.3740, доп. стоимость=3209.35
  1 сервер (PS) к chain4 (μ=1.1979, α=0.5000): E[T]=0.3751, доп. стоимость=1717.50
  1 сервер (FIFO) к chain4 (μ=1.1979, α=0.5000): E[T]=0.3751, доп. стоимость=872.74

=== Сценарий 2: два сервера ===
  Подсценарий 2a: два сервера к одному узлу, бюджет поровну
  2 сервера к chain1: PS(μ=1.6995,α=0.3333) и PS(μ=1.6995,α=0.3333): E[T]=1.6354, доп. стоимость=5165.61 (бюджет 5165.60)
  2 сервера к chain1: FIFO(μ=2.1130,α=0.3333) и FIFO(μ=2.1130,α=0.3333): E[T]=0.9243, доп. стоимост

## Теоретический расчёт для выбранной конфигурации

In [15]:
lambda_eff_new, _, _, _, _ = solve_traffic(new_config['servers'], new_config['transitions'])
metrics_new, E_T_sys_new, _ = compute_metrics(new_config['servers'], lambda_eff_new)

print("=== Результаты после оптимизации ===")
for sid in metrics_new:
    m = metrics_new[sid]
    print(f"{sid}: λ={m['lambda']:.4f}, ρ={m['rho']:.4f}, E[T]={m['E_T']:.4f}, вклад={m['contrib']:.4f}")
print(f"\nНовое теоретическое среднее время E[T_sys] = {E_T_sys_new:.4f}")

=== Результаты после оптимизации ===
chain1: λ=3.7169, ρ=0.5192, E[T]=0.2905, вклад=0.3016
chain2: λ=0.2212, ρ=0.0935, E[T]=0.4660, вклад=0.0288
chain3: λ=0.5870, ρ=0.0965, E[T]=0.1819, вклад=0.0298
chain4: λ=0.0206, ρ=0.0172, E[T]=0.8494, вклад=0.0049
chain4_p1: λ=0.0206, ρ=0.0098, E[T]=0.4779, вклад=0.0028
chain4_p2: λ=0.0206, ρ=0.0098, E[T]=0.4779, вклад=0.0028

Новое теоретическое среднее время E[T_sys] = 0.3706


## Новый YAML

In [16]:
def convert_to_python_types(obj):
    if isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, dict):
        return {key: convert_to_python_types(value) for key, value in obj.items()}
    elif isinstance(obj, (list, tuple)):
        return [convert_to_python_types(item) for item in obj]
    else:
        return obj

new_config_clean = convert_to_python_types(new_config)
for t in new_config_clean['transitions']:
    t['probability'] = round(t['probability'], 8)

import yaml
new_yaml_str = yaml.dump(new_config_clean, sort_keys=False, allow_unicode=True, default_flow_style=False)
print("=== Новая конфигурация (YAML) ===\n")
print(new_yaml_str)

with open('optimized_config.yaml', 'w') as f:
    f.write(new_yaml_str)
print("Конфигурация сохранена в 'optimized_config.yaml'")

=== Новая конфигурация (YAML) ===

servers:
- id: chain1
  mu: 7.159457094969063
  queue_type: FIFO
  lambda: 3.579728547484531
  Cost: 26628.9
  Jobs: 3751
  Avg_queue_time: 0.145195
  Avg_service_time: 0.139562
  Utilization: 0.523677
- id: chain2
  mu: 2.367157068548567
  queue_type: PS
  Cost: 3801.72
  Jobs: 225
  Avg_queue_time: 0.0462857
  Avg_service_time: 0.397827
  Utilization: 0.089511
- id: chain3
  mu: 6.084057042335131
  queue_type: FIFO
  Cost: 19507.9
  Jobs: 603
  Avg_queue_time: 0.0264107
  Avg_service_time: 0.173573
  Utilization: 0.104665
- id: chain4
  mu: 1.197904448724397
  queue_type: PS
  Cost: 1717.49
  Jobs: 57
  Avg_queue_time: 0.0160039
  Avg_service_time: 0.825079
  Utilization: 0.0470295
- id: chain4_p1
  mu: 2.112953837898736
  queue_type: FIFO
- id: chain4_p2
  mu: 2.112953837898736
  queue_type: FIFO
transitions:
- from: chain1
  to: exit
  probability: 0.79986142
- from: chain1
  to: chain3
  probability: 0.15793907
- from: chain1
  to: chain2
  proba

## Проверка корректности переходов

In [17]:
for srv in new_config_clean['servers']:
    sid = srv['id']
    outgoing = [t['probability'] for t in new_config_clean['transitions'] if t['from'] == sid]
    total = sum(outgoing)
    print(f"{sid}: сумма вероятностей переходов = {total:.6f} (exit = {1-total:.6f})")

chain1: сумма вероятностей переходов = 1.000000 (exit = 0.000000)
chain2: сумма вероятностей переходов = 1.000000 (exit = -0.000000)
chain3: сумма вероятностей переходов = 1.000000 (exit = 0.000000)
chain4: сумма вероятностей переходов = 1.000000 (exit = 0.000000)
chain4_p1: сумма вероятностей переходов = 1.000000 (exit = 0.000000)
chain4_p2: сумма вероятностей переходов = 1.000000 (exit = 0.000000)


## Итоговый вывод

In [18]:
print("="*60)
print("ОПТИМИЗАЦИЯ ЗАВЕРШЕНА")
print("="*60)
print(f"Исходная стоимость системы: {total_cost:.2f}")
print(f"Бюджет на новые серверы: {budget:.2f} (10%)")
print(f"Выбранный вариант: {best_desc}")
# Оценим фактическую стоимость добавленных серверов
added_actual = evaluate_cost(new_config) - total_cost
print(f"Фактическая дополнительная стоимость: {added_actual:.2f}")
print(f"Теоретическое улучшение E[T_sys]: {E_T_sys:.4f} → {E_T_sys_new:.4f} ({(E_T_sys - E_T_sys_new)/E_T_sys*100:.2f}%)")
print("Новая конфигурация сохранена в 'optimized_config.yaml'")

ОПТИМИЗАЦИЯ ЗАВЕРШЕНА
Исходная стоимость системы: 51656.00
Бюджет на новые серверы: 5165.60 (10%)
Выбранный вариант: 2 сервера к chain4: FIFO(μ=2.1130,α=0.3333) и FIFO(μ=2.1130,α=0.3333)
Фактическая дополнительная стоимость: 5165.61
Теоретическое улучшение E[T_sys]: 0.3755 → 0.3706 (1.28%)
Новая конфигурация сохранена в 'optimized_config.yaml'


![img_1.png](img_1.png)

## Реальный результат: 
```
Simulation results (max time: 1000):
Server 'chain1'
  Type: FIFO
  Cost: 14899.8
  Jobs: 2239
  Avg queue time: 0.113014
  Avg service time: 0.180828
  Utilization: 0.404874
Server 'chain1_p'
  Type: PS
  Cost: 8134.49
  Jobs: 686
  Avg queue time: 0.0509475
  Avg service time: 0.261501
  Utilization: 0.17939
Server 'chain2'
  Type: PS
  Cost: 18038.5
  Jobs: 677
  Avg queue time: 0.0198502
  Avg service time: 0.171488
  Utilization: 0.11615
Server 'chain3'
  Type: FIFO
  Cost: 11347.3
  Jobs: 227
  Avg queue time: 0.00914131
  Avg service time: 0.20207
  Utilization: 0.0465772
Server 'chain4'
  Type: PS
  Cost: 33999.6
  Jobs: 873
  Avg queue time: 0.016838
  Avg service time: 0.129713
  Utilization: 0.113239

Total system cost: 86419.7
Total tasks: 2558
Mean time: 0.421695

Linear hist:
*                                       
*                                       
*                                       
*                                       
* *                                     
* *                                     
* *                                     
* * *                                   
* * *                                   
* * *                                   
* * *                                   
* * * *                                 
* * * *                                 
* * * *                                 
* * * * *                               
* * * * *                               
* * * * * *                             
* * * * * * *                           
* * * * * * *                           
* * * * * * * * * *                     
---------------------------------------
0.00                                3.05

Log hist:
  * *                                   
* * *                                   
* * *                                   
* * *   *                               
* * *   *                               
* * *   *                               
* * * * * *                             
* * * * * *                             
* * * * * *                             
* * * * * * *                           
* * * * * * *                           
* * * * * * * *                         
* * * * * * * *                         
* * * * * * * * *                       
* * * * * * * * *                       
* * * * * * * * * *                     
* * * * * * * * * * *                   
* * * * * * * * * * *                   
* * * * * * * * * * * * *               
* * * * * * * * * * * * * * * *         
---------------------------------------
0.00                                2.02
```

Есть небольшой вылет за бюджет, а значит модель расчёта стоимости неточная